#LOADING FILES

##LIBRARIES AND PACKAGES

In [0]:
%pip install sec-edgar-downloader
%pip install camelot-py[cv]
%pip install tabula-py
%pip install pdfplumber
%pip install pdfplumber pandas


In [0]:
from sec_edgar_downloader import Downloader
from bs4 import BeautifulSoup
from pathlib import Path
import pandas as pd
import pdfplumber
import re
from pyspark.sql import SparkSession
import requests
from sec_utils import *

## PDF EXTRACTION AND SAVING

In [0]:
ticker_to_cik = {
    "AAPL": "0000320193",
    "MSFT": "0000789019"
}

In [0]:
BASE = "https://data.sec.gov/api/xbrl/companyfacts"
HEADERS = {
    "User-Agent": "Alejandro Lopez (al.lopez.moreira@gmail.com)"
}

def get_company_facts(cik):
    cik = cik.zfill(10)
    url = f"{BASE}/CIK{cik}.json"
    res = requests.get(url, headers=HEADERS)
    res.raise_for_status()
    return res.json()

def extract_metric(data, concept):
    try:
        facts = data["facts"]["us-gaap"][concept]["units"]
        
        if "USD" in facts:
            obs = facts["USD"]
        else:
            unit = list(facts.keys())[0]
            obs = facts[unit]
            
        return obs
    except KeyError:
        return []

def clean_observations(obs):
    cleaned = []
    
    for x in obs:
        if x.get("form") in ["10-K", "10-Q"]:
            cleaned.append({
                "value": x.get("val"),
                "date": x.get("end"),
                "form": x.get("form"),
                "filed": x.get("filed")
            })
    
    return cleaned

def to_df(cleaned, metric_name):
    df = pd.DataFrame(cleaned)
    
    if df.empty:
        return df
    
    df["metric"] = metric_name
    df["date"] = pd.to_datetime(df["date"])
    df["filed"] = pd.to_datetime(df["filed"])
    
    return df

In [0]:
data = get_company_facts("0000320193")  # Apple

obs = extract_metric(data, "Revenues")
cleaned = clean_observations(obs)
df_rev = to_df(cleaned, "revenue")

df_rev = df_rev.sort_values("filed").drop_duplicates(
    subset=["date"], keep="last"
)

df_rev.sort_values("date").tail()

In [0]:
df_rev.tail(10)